# Heterogeneous message-direction sweep (`gs_hmd`)

This notebook analyses the `heterogenous_message_direction` folder: the full
3-by-3 factorial of generator and load attachment directions in the direct
heterogeneous graph, with three seeds each (27 runs).

Every run is otherwise identical -- `heterogeneous` graph, no input
preprocessing, GINE, mean pooling, no substation edges or nodes, and a
15M-step budget -- so `generator_direction` and `load_direction` are the only
screened factors.

The baseline is `bidirectional / bidirectional` (`gbi_lbi`), and the main
views are:

- aggregated episodic-survival curves for all nine direction pairs;
- the same curves expressed as a **difference from the baseline**;
- absolute and baseline-relative heatmaps over the direction grid.

In [ ]:
from pathlib import Path
import importlib
import sys
import tomllib

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError(
        "Could not locate Topology_Task/analysis/metrics/helpers"
    )

import wandb_metrics as wm
wm = importlib.reload(wm)
import survival_comparison as sc
sc = importlib.reload(sc)
print("wandb_metrics:", wm.__file__)
print("survival_comparison:", sc.__file__)
print("task directory:", wm.TASK_DIR)

## Analysis controls

`COMPARISON_BUDGET_STEPS = None` compares every run at the largest step that
**all** selected runs reached. That matters here: these runs are configured for
15M steps but stop early when `time_limit` (2880 minutes) expires, so the
achieved endpoint varies per run and per cluster. Set an explicit value to pin
the comparison point instead.

`HEATMAP_LAST_N_TEST_EVALS` controls the heatmap statistic: the mean of each
run's final N **unsmoothed** test evaluations.

In [ ]:
USE_LOCAL_CACHE_ONLY = True
SMOOTH_WINDOW = 5
HEATMAP_LAST_N_TEST_EVALS = 7
TARGET_BUDGET_STEPS = 15_000_000
COMPARISON_BUDGET_STEPS = None

HMD_DIR = (
    wm.TASK_DIR / "configs" / "gnn_graph_screening"
    / "heterogenous_message_direction"
)

# Short codes matching the config file names.
DIRECTION_CODES = {
    "bidirectional": "bi",
    "asset_to_busbar": "a2b",
    "busbar_to_asset": "b2a",
}
DIRECTION_ORDER = ["bidirectional", "asset_to_busbar", "busbar_to_asset"]

BASELINE_GENERATOR_DIRECTION = "bidirectional"
BASELINE_LOAD_DIRECTION = "bidirectional"
BASELINE_LABEL = "g=bi · l=bi"

FACTOR_COLUMNS = ["generator_direction", "load_direction"]

print("config folder:", HMD_DIR)
print("baseline:", BASELINE_LABEL)

## Build the run catalog from TOML

Settings are read from the config files rather than parsed out of file names,
so the catalog stays correct if a file is renamed or reseeded.

In [ ]:
def direction_code(value):
    return DIRECTION_CODES.get(str(value), str(value))


def config_record(path):
    with path.open("rb") as file:
        config = tomllib.load(file)
    args = config["args"]
    run = config.get("run", {})
    record = {
        "config_path": str(path.relative_to(wm.TASK_DIR)),
        "config": path.name,
        "run_name": str(run.get("name", path.stem)),
        "seed": int(args.get("seed", 0)),
        "declared_cuda": bool(args.get("cuda", False)),
        "graph_type": str(args.get("gnn_graph_type", "bus")),
        "encoder": str(args.get("gnn_type", "gine")),
        "generator_direction": str(
            args.get("gnn_generator_edge_direction", "bidirectional")
        ),
        "load_direction": str(
            args.get("gnn_load_edge_direction", "bidirectional")
        ),
        "line_direction": str(
            args.get("gnn_line_node_edge_direction", "bidirectional")
        ),
        "summary_direction": str(
            args.get("gnn_summary_edge_direction", "bidirectional")
        ),
        "configured_steps": int(args.get("total_timesteps", TARGET_BUDGET_STEPS)),
        "time_limit_minutes": float(args.get("time_limit", np.nan)),
    }
    record["generator_code"] = direction_code(record["generator_direction"])
    record["load_code"] = direction_code(record["load_direction"])
    record["direction_label"] = (
        f'g={record["generator_code"]} · l={record["load_code"]}'
    )
    record["factor_key"] = "|".join(
        str(record[column]) for column in FACTOR_COLUMNS
    )
    record["is_baseline"] = (
        record["generator_direction"] == BASELINE_GENERATOR_DIRECTION
        and record["load_direction"] == BASELINE_LOAD_DIRECTION
    )
    return record


if not HMD_DIR.exists():
    raise FileNotFoundError(f"Missing config folder: {HMD_DIR}")

config_catalog = pd.DataFrame(
    [config_record(path) for path in sorted(HMD_DIR.glob("*.toml"))]
)
if config_catalog.empty:
    raise RuntimeError(f"No TOML files were found in {HMD_DIR}.")

print(f"Cataloged {len(config_catalog)} declared runs.")

# Anything unexpectedly non-constant would confound the direction screen.
held_constant = [
    "graph_type",
    "encoder",
    "line_direction",
    "summary_direction",
    "configured_steps",
]
for column in held_constant:
    values = sorted(config_catalog[column].astype(str).unique())
    flag = "" if len(values) == 1 else "  <-- NOT CONSTANT"
    print(f"  {column}: {values}{flag}")

display(
    config_catalog.pivot_table(
        index="generator_code",
        columns="load_code",
        values="seed",
        aggfunc="count",
        fill_value=0,
    ).rename_axis(index="generator", columns="load")
)

## Load the W&B histories

Only the 27 run names declared by the folder are selected. The compute backend
is taken from the effective downloaded run configuration (`cuda=True` means the
IZAR/GPU cluster), because the launch script can override the TOML.

In [ ]:
requested_run_names = config_catalog["run_name"].drop_duplicates().tolist()
requested_run_name_set = set(requested_run_names)
wm.configure_run_filter_from_names(requested_run_names)
data = wm.load_wandb_data(use_local_cache_only=USE_LOCAL_CACHE_ONLY)

runs_df = data.runs_df.copy()
history_df = data.history_df.copy()
if not history_df.empty:
    history_df = history_df[
        history_df["run_name"].astype(str).isin(requested_run_name_set)
    ].copy()
if "name" in runs_df:
    runs_df = runs_df[
        runs_df["name"].astype(str).isin(requested_run_name_set)
    ].copy()


def parse_optional_bool(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {"true", "1", "yes", "y"}:
            return True
        if normalized in {"false", "0", "no", "n"}:
            return False
    return bool(value)


if "cuda" in runs_df and "name" in runs_df:
    runtime_backend = runs_df[["name", "cuda"]].copy()
    runtime_backend["runtime_cuda"] = runtime_backend["cuda"].map(
        parse_optional_bool
    )
    runtime_backend = (
        runtime_backend.dropna(subset=["runtime_cuda"])
        .drop_duplicates("name", keep="last")
        .rename(columns={"name": "run_name"})[["run_name", "runtime_cuda"]]
    )
    config_catalog = config_catalog.merge(
        runtime_backend, on="run_name", how="left"
    )
else:
    config_catalog["runtime_cuda"] = np.nan
config_catalog["runtime_cuda"] = config_catalog["runtime_cuda"].where(
    config_catalog["runtime_cuda"].notna(), config_catalog["declared_cuda"]
)
config_catalog["runtime_cuda"] = config_catalog["runtime_cuda"].astype(bool)
config_catalog["compute_backend"] = np.where(
    config_catalog["runtime_cuda"],
    "IZAR (GPU)",
    "JED (CPU)",
)

found_run_names = set(history_df.get("run_name", pd.Series(dtype=str)))
print(
    f"Loaded histories for {len(found_run_names)} / "
    f"{len(requested_run_names)} declared runs."
)
missing_run_names = sorted(requested_run_name_set - found_run_names)
if missing_run_names:
    print("Missing histories:")
    for name in missing_run_names:
        print("  ", name)

## Coverage

Because `time_limit` can stop a run before the 15M-step budget, check how far
each run actually got before trusting any endpoint comparison. A wide spread
here -- especially one that lines up with the compute backend -- is itself a
finding.

In [ ]:
if history_df.empty:
    raise RuntimeError(
        "No history was loaded. Download the gs_hmd runs first, or set "
        "USE_LOCAL_CACHE_ONLY = False."
    )

observed_progress = (
    history_df.groupby("run_name", as_index=False)["step"]
    .max()
    .rename(columns={"step": "observed_steps"})
)
coverage = config_catalog.merge(observed_progress, on="run_name", how="left")
coverage["observed_steps_m"] = coverage["observed_steps"] / 1_000_000
coverage["completion_pct"] = (
    100 * coverage["observed_steps"] / coverage["configured_steps"]
)
coverage["has_history"] = coverage["observed_steps"].notna()

analysis_catalog = coverage[coverage["has_history"]].copy()

with pd.option_context("display.max_colwidth", None):
    display(
        coverage.sort_values(["generator_code", "load_code", "seed"])[
            [
                "direction_label",
                "seed",
                "run_name",
                "compute_backend",
                "observed_steps_m",
                "completion_pct",
                "has_history",
            ]
        ].round(2)
    )

print("Observed-step spread by backend:")
display(
    analysis_catalog.groupby("compute_backend", as_index=False).agg(
        runs=("run_name", "nunique"),
        min_steps_m=("observed_steps_m", "min"),
        median_steps_m=("observed_steps_m", "median"),
        max_steps_m=("observed_steps_m", "max"),
    ).round(2)
)

progress_plot = px.bar(
    coverage.sort_values("observed_steps_m"),
    x="observed_steps_m",
    y="run_name",
    color="compute_backend",
    orientation="h",
    hover_data=["direction_label", "seed"],
    title="gs_hmd run coverage",
    labels={
        "observed_steps_m": "Observed environment steps (millions)",
        "run_name": "Run",
    },
    height=max(600, 20 * len(coverage)),
)
progress_plot.add_vline(
    x=TARGET_BUDGET_STEPS / 1_000_000,
    line_dash="dash",
    annotation_text="15M target",
)
progress_plot.show()

## Extract test episodic-survival curves

`extract_survival_curves` picks the first available test-survival metric per
run and converts it to percentage points. Smoothing is left at 1 here so each
plot below can choose its own window.

In [ ]:
survival_long = sc.extract_survival_curves(
    history_df,
    catalog=analysis_catalog,
    split="test",
    smooth=1,
)
if survival_long.empty:
    raise RuntimeError("No test episodic-survival history was found.")

print("metrics used:", sorted(survival_long["metric"].unique()))
print(f"{survival_long['run_name'].nunique()} runs with survival curves")

max_survival_steps = survival_long.groupby("run_name")["step"].max()
automatic_common_budget = int(max_survival_steps.min())
comparison_budget = int(
    COMPARISON_BUDGET_STEPS
    if COMPARISON_BUDGET_STEPS is not None
    else automatic_common_budget
)
print(
    "Comparison budget:",
    f"{comparison_budget / 1_000_000:.3f}M steps",
    "(automatic common budget)"
    if COMPARISON_BUDGET_STEPS is None
    else "(user selected)",
)
if COMPARISON_BUDGET_STEPS is None:
    slowest = max_survival_steps.idxmin()
    print(f"  set by the shortest run: {slowest}")

## Per-run endpoint statistics

- `last_n_test_survival_pct`: mean of the final N **unsmoothed** test
  evaluations (the heatmap statistic);
- `comparison_survival_pct`: smoothed survival at the common budget above.

In [ ]:
def endpoint_at_budget(frame, budget):
    if frame["step"].max() < budget:
        return np.nan
    eligible = frame[frame["step"] <= budget]
    if eligible.empty:
        return np.nan
    smoothed = (
        eligible["raw_survival_pct"]
        .rolling(SMOOTH_WINDOW, min_periods=1)
        .mean()
    )
    return float(smoothed.iloc[-1])


endpoint_rows = []
for run_name, frame in survival_long.groupby("run_name", sort=False):
    frame = frame.sort_values("step")
    tail = frame.tail(HEATMAP_LAST_N_TEST_EVALS)
    endpoint_rows.append(
        {
            "run_name": run_name,
            "last_eval_step": frame["step"].max(),
            "n_test_evals": len(frame),
            "last_n_test_evals": len(tail),
            "last_n_test_survival_pct": float(tail["raw_survival_pct"].mean()),
            "comparison_survival_pct": endpoint_at_budget(
                frame, comparison_budget
            ),
            "target_survival_pct": endpoint_at_budget(
                frame, TARGET_BUDGET_STEPS
            ),
        }
    )

endpoint_df = analysis_catalog.merge(
    pd.DataFrame(endpoint_rows), on="run_name", how="left"
)
endpoint_df["last_eval_step_m"] = endpoint_df["last_eval_step"] / 1_000_000

with pd.option_context("display.max_colwidth", None):
    display(
        endpoint_df.sort_values(
            "last_n_test_survival_pct", ascending=False, na_position="last"
        )[
            [
                "direction_label",
                "seed",
                "compute_backend",
                "last_eval_step_m",
                "last_n_test_evals",
                "last_n_test_survival_pct",
                "comparison_survival_pct",
                "config_path",
            ]
        ].round(2)
    )

## Aggregated survival curves

One curve per direction pair, aggregated over its three seeds. Thin lines are
the individual seeds; the band is one standard deviation across them. The
baseline is drawn in grey.

In [ ]:
DIRECTION_COLORS = {
    BASELINE_LABEL: "#6b7280",
    "g=bi · l=a2b": "#1f77b4",
    "g=bi · l=b2a": "#17becf",
    "g=a2b · l=bi": "#2ca02c",
    "g=a2b · l=a2b": "#d62728",
    "g=a2b · l=b2a": "#ff7f0e",
    "g=b2a · l=bi": "#9467bd",
    "g=b2a · l=a2b": "#8c564b",
    "g=b2a · l=b2a": "#e377c2",
}

absolute_figure = sc.plot_survival_comparison(
    survival_long,
    group_by=["generator_code", "load_code"],
    label_by="direction_label",
    smooth=SMOOTH_WINDOW,
    uncertainty="std",
    min_members=1,
    show_members=True,
    colors=DIRECTION_COLORS,
    highlight=[BASELINE_LABEL],
    budget_step=comparison_budget,
    title=(
        "gs_hmd: test episodic survival by generator / load message direction"
    ),
    width=1450,
    height=700,
)
absolute_figure.show()

### Small multiples by generator direction

The same data faceted so each panel holds one generator direction and compares
the three load directions inside it.

In [ ]:
faceted_figure = sc.plot_survival_comparison(
    survival_long,
    group_by=["generator_code", "load_code"],
    label_by=lambda row: f'l={row["load_code"]}',
    facet_by="generator_direction",
    facet_order=DIRECTION_ORDER,
    smooth=SMOOTH_WINDOW,
    uncertainty="std",
    min_members=1,
    budget_step=comparison_budget,
    title="gs_hmd: load direction within each generator direction",
    width=1450,
    height=520,
    ncols=3,
)
faceted_figure.show()

## Difference from the `gbi_lbi` baseline

Each curve is the aggregated direction pair minus the aggregated baseline at
the same step, so the baseline is the flat zero line. Values above zero mean
that direction pair is outperforming bidirectional/bidirectional at that point
in training. The band combines the uncertainty of both aggregated curves, so a
band overlapping zero means the difference is not resolved by three seeds.

In [ ]:
difference_figure = sc.plot_survival_comparison(
    survival_long,
    group_by=["generator_code", "load_code"],
    label_by="direction_label",
    smooth=SMOOTH_WINDOW,
    uncertainty="ci95",
    min_members=1,
    comparison="difference",
    baseline={
        "generator_code": DIRECTION_CODES[BASELINE_GENERATOR_DIRECTION],
        "load_code": DIRECTION_CODES[BASELINE_LOAD_DIRECTION],
    },
    colors=DIRECTION_COLORS,
    budget_step=comparison_budget,
    y_range=None,
    title=(
        "gs_hmd: survival difference from the bidirectional / bidirectional "
        "baseline"
    ),
    y_title="Survival difference from baseline (pp)",
    width=1450,
    height=700,
)
difference_figure.add_hline(y=0.0, line_dash="dot", line_color="#6b7280")
difference_figure.show()

> **Do not add `facet_by=` to a `comparison="difference"` call.**
> `survival_comparison._difference_from_baseline` inner-joins the baseline on
> `[facet_column, step]`, and this baseline only exists inside the
> `generator_direction == "bidirectional"` facet, so faceting silently drops
> every other panel instead of raising. Use the un-faceted difference plot
> above, or the baseline-relative heatmap below, for per-cell deltas.

## Direction heatmaps

Cell statistic: for each run, the mean of its final
`HEATMAP_LAST_N_TEST_EVALS` **unsmoothed** test evaluations; runs sharing a
cell are then averaged so every run has equal weight. Runs with fewer than
that many evaluations are excluded.

Three views are produced:

1. **absolute** mean survival per direction pair;
2. **difference from the baseline** cell, on a diverging scale;
3. **seed count** per cell, to confirm each cell really has three runs.

Each heatmap is followed by the exact configs behind it.

In [ ]:
HEATMAP_VALUE_COLUMN = "last_n_test_survival_pct"
heatmap_endpoint_df = (
    endpoint_df[
        endpoint_df["last_n_test_evals"] >= HEATMAP_LAST_N_TEST_EVALS
    ]
    .dropna(subset=[HEATMAP_VALUE_COLUMN])
    .copy()
)
excluded = len(endpoint_df) - len(heatmap_endpoint_df)
if excluded:
    print(
        f"Excluded {excluded} run(s) with fewer than "
        f"{HEATMAP_LAST_N_TEST_EVALS} test evaluations."
    )

DIRECTION_CODE_ORDER = [DIRECTION_CODES[name] for name in DIRECTION_ORDER]


def direction_table(frame, value_column, aggfunc="mean"):
    table = frame.pivot_table(
        index="generator_code",
        columns="load_code",
        values=value_column,
        aggfunc=aggfunc,
    )
    return table.reindex(
        index=DIRECTION_CODE_ORDER, columns=DIRECTION_CODE_ORDER
    ).rename_axis(index="generator", columns="load")


def show_direction_heatmap(
    table,
    title,
    color_label,
    colorscale="Viridis",
    zmin=None,
    zmax=None,
    zmid=None,
    text_format=".1f",
    source_df=None,
):
    if table.empty or table.notna().sum().sum() == 0:
        print(f"No data available for: {title}")
        return None
    print(title)
    display(table.round(2))
    kwargs = dict(
        text_auto=text_format,
        aspect="auto",
        color_continuous_scale=colorscale,
        labels={
            "x": "Load direction",
            "y": "Generator direction",
            "color": color_label,
        },
        title=title,
    )
    if zmin is not None:
        kwargs["zmin"] = zmin
    if zmax is not None:
        kwargs["zmax"] = zmax
    if zmid is not None:
        kwargs["color_continuous_midpoint"] = zmid
    figure = px.imshow(table, **kwargs)
    figure.show()
    if source_df is not None:
        detail = (
            source_df[
                [
                    "generator_code",
                    "load_code",
                    "seed",
                    "compute_backend",
                    "run_name",
                    "config_path",
                    HEATMAP_VALUE_COLUMN,
                ]
            ]
            .sort_values(["generator_code", "load_code", "seed"])
            .reset_index(drop=True)
        )
        print(f"Configs used for: {title}")
        with pd.option_context("display.max_colwidth", None):
            display(detail.round(2))
    return figure


absolute_table = direction_table(heatmap_endpoint_df, HEATMAP_VALUE_COLUMN)
show_direction_heatmap(
    absolute_table,
    "Generator × load message direction — mean final-"
    f"{HEATMAP_LAST_N_TEST_EVALS} test survival",
    "Mean survival (%)",
    colorscale="Viridis",
    zmin=0,
    zmax=100,
    source_df=heatmap_endpoint_df,
)

In [ ]:
baseline_code = (
    DIRECTION_CODES[BASELINE_GENERATOR_DIRECTION],
    DIRECTION_CODES[BASELINE_LOAD_DIRECTION],
)
baseline_value = absolute_table.loc[baseline_code[0], baseline_code[1]]
if pd.isna(baseline_value):
    raise ValueError(
        "The baseline cell has no value, so a relative heatmap is undefined."
    )
print(f"Baseline ({BASELINE_LABEL}) = {baseline_value:.2f}%")

delta_table = absolute_table - baseline_value
delta_limit = float(np.nanmax(np.abs(delta_table.to_numpy())))
show_direction_heatmap(
    delta_table,
    f"Difference from the {BASELINE_LABEL} baseline",
    "Survival difference (pp)",
    colorscale="RdBu",
    zmin=-delta_limit,
    zmax=delta_limit,
    zmid=0.0,
    text_format="+.1f",
)

In [ ]:
seed_count_table = direction_table(
    heatmap_endpoint_df, "run_name", aggfunc="nunique"
)
show_direction_heatmap(
    seed_count_table,
    "Runs contributing to each cell",
    "Runs",
    colorscale="Blues",
    zmin=0,
    text_format="d",
)

spread_table = direction_table(
    heatmap_endpoint_df, HEATMAP_VALUE_COLUMN, aggfunc="std"
)
show_direction_heatmap(
    spread_table,
    "Seed-to-seed standard deviation per cell",
    "Std across seeds (pp)",
    colorscale="Oranges",
    zmin=0,
)

## Direction-pair ranking

Seed-aggregated ranking at the common budget, with the baseline delta attached.
A pair only beats the baseline convincingly if `mean_minus_baseline` is
positive and comfortably larger than `std_survival_pct`.

In [ ]:
valid_endpoint_df = endpoint_df.dropna(
    subset=["comparison_survival_pct"]
).copy()

ranking = (
    valid_endpoint_df.groupby(
        ["generator_code", "load_code", "direction_label"],
        as_index=False,
    )
    .agg(
        mean_survival_pct=("comparison_survival_pct", "mean"),
        std_survival_pct=("comparison_survival_pct", "std"),
        min_survival_pct=("comparison_survival_pct", "min"),
        max_survival_pct=("comparison_survival_pct", "max"),
        seeds=("seed", "nunique"),
        backends=("compute_backend", lambda x: ", ".join(sorted(set(x)))),
    )
    .sort_values("mean_survival_pct", ascending=False)
)

baseline_mask = (
    ranking["generator_code"].eq(baseline_code[0])
    & ranking["load_code"].eq(baseline_code[1])
)
if not baseline_mask.any():
    raise ValueError("The baseline pair is missing from the ranking.")
baseline_mean = float(ranking.loc[baseline_mask, "mean_survival_pct"].iloc[0])
ranking["mean_minus_baseline"] = ranking["mean_survival_pct"] - baseline_mean

display(ranking.round(2))

rank_plot = px.bar(
    ranking.sort_values("mean_minus_baseline"),
    x="mean_minus_baseline",
    y="direction_label",
    orientation="h",
    color="mean_minus_baseline",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
    hover_data=["mean_survival_pct", "std_survival_pct", "seeds"],
    title=(
        f"Survival at the {comparison_budget / 1_000_000:.3f}M-step budget, "
        f"relative to {BASELINE_LABEL}"
    ),
    labels={
        "mean_minus_baseline": "Difference from baseline (pp)",
        "direction_label": "Direction pair",
    },
    height=520,
)
rank_plot.add_vline(x=0.0, line_dash="dot")
rank_plot.show()

## Marginal direction effects

Averaging over the other relation. These are exploratory marginals, not causal
estimates, but the design here is a balanced full factorial with equal seeds
per cell, so they are more trustworthy than the equivalent tables in the
combined Stage 1 notebook.

In [ ]:
for factor, label in [
    ("generator_direction", "Generator direction"),
    ("load_direction", "Load direction"),
]:
    summary = (
        valid_endpoint_df.groupby(factor, as_index=False)
        .agg(
            mean_survival_pct=("comparison_survival_pct", "mean"),
            std_survival_pct=("comparison_survival_pct", "std"),
            runs=("run_name", "nunique"),
        )
        .sort_values("mean_survival_pct", ascending=False)
    )
    print(label)
    display(summary.round(2))

## Optional CSV export

In [ ]:
EXPORT_TABLES = False

if EXPORT_TABLES:
    export_dir = wm.TASK_DIR / "outputs" / "gs_hmd_direction_summary"
    export_dir.mkdir(parents=True, exist_ok=True)
    coverage.to_csv(export_dir / "coverage.csv", index=False)
    endpoint_df.to_csv(export_dir / "run_endpoints.csv", index=False)
    ranking.to_csv(export_dir / "direction_ranking.csv", index=False)
    absolute_table.to_csv(export_dir / "heatmap_absolute.csv")
    delta_table.to_csv(export_dir / "heatmap_baseline_delta.csv")
    print("Saved tables under", export_dir)
else:
    print("Set EXPORT_TABLES = True to write CSVs.")